In [2]:
!pip install dagshub mlflow -q

import dagshub
import mlflow
import pandas as pd
import numpy as np
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
token = user_secrets.get_secret("DAGSHUB_TOKEN")

dagshub.init(repo_owner='kvizhmena11', repo_name='ML-assignment2-IEEE-Fraud-Detection', mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow")
mlflow.set_experiment("RandomForest_Training")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=ac2b0c0c-f8c3-4aac-aa6b-b157105621ff&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=574808dc8f7a260716cebf3b8ae541c1495c3c4bea0793622e6ae7bbe4567647




Output()

Accessing as kvizhmena11

Initialized MLflow to track repo "kvizhmena11/ML-assignment2-IEEE-Fraud-Detection"

Repository kvizhmena11/ML-assignment2-IEEE-Fraud-Detection initialized!

2026/05/07 13:13:12 INFO mlflow.tracking.fluent: Experiment with name 'RandomForest_Training' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/88dc80b537b54894a8b47c022d1345cc', creation_time=1778159592218, experiment_id='3', last_update_time=1778159592218, lifecycle_stage='active', name='RandomForest_Training', tags={}, trace_location=None, workspace='default'>

# 🧹 Cleaning

In [3]:
def reduce_mem_usage(df, verbose=True):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(
            end_mem, 100 * (start_mem - end_mem) / start_mem))
    return df

In [10]:
mlflow.end_run()
base_path = '/kaggle/input/competitions/ieee-fraud-detection/'

with mlflow.start_run(run_name="RandomForest_Cleaning"):
    train_transaction = pd.read_csv(base_path + 'train_transaction.csv')
    train_identity = pd.read_csv(base_path + 'train_identity.csv')
    train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
    del train_transaction, train_identity

    mlflow.log_metric("initial_nan_count", train.isnull().sum().sum())
    mlflow.log_metric("initial_rows", len(train))
    mlflow.log_metric("initial_cols", len(train.columns))

    train = reduce_mem_usage(train)

    # Fill numeric NaNs
    num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
    for col in num_cols:
        if col not in ['isFraud', 'TransactionID']:
            train[col] = train[col].fillna(-999)

    mlflow.log_param("nan_fill_strategy", "numeric=-999")
    mlflow.log_param("rows_after_cleaning", len(train))
    mlflow.log_param("cols_after_cleaning", len(train.columns))
    print(f"Dataset shape: {train.shape}")

/tmp/ipykernel_57/763760899.py:19: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_57/763760899.py:19: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_57/763760899.py:19: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_57/763760899.py:19: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_57/763760899.py:19: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_57/763760899.py:19: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_57/763760899.py:19: RuntimeWarning: overflow enco

Mem. usage decreased to 645.97 Mb (67.0% reduction)
Dataset shape: (590540, 434)
🏃 View run RandomForest_Cleaning at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3/runs/d56da7d0ae0c4eb38aaaee0addb8d899
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3


# ⚙️ Feature Engineering

In [11]:
with mlflow.start_run(run_name="RandomForest_Feature_Engineering"):

    emails = {'gmail': 'google', 'att.net': 'at&t', 'twc.com': 'spectrum',
              'outlook.com': 'microsoft', 'msn.com': 'microsoft',
              'hotmail.com': 'microsoft', 'live.com': 'microsoft',
              'icloud.com': 'apple', 'mac.com': 'apple', 'me.com': 'apple'}

    for c in ['P_emaildomain', 'R_emaildomain']:
        train[c + '_bin'] = train[c].map(emails)
        train[c + '_suffix'] = train[c].map(lambda x: str(x).split('.')[-1])

    train['Trans_hour'] = (train['TransactionDT'] // 3600) % 24
    train['Trans_day_of_week'] = (train['TransactionDT'] // (3600 * 24)) % 7

    for col in ['card1', 'card2', 'card3', 'card5']:
        if col in train.columns:
            freq = train[col].astype(np.float32).value_counts(normalize=True)
            train[f'{col}_freq'] = train[col].astype(np.float32).map(freq)

    train['TransactionAmt_log'] = np.log1p(train['TransactionAmt'])
    train['TransactionAmt_decimal'] = ((train['TransactionAmt'].astype(np.float32) - train['TransactionAmt'].astype(int)) * 1000).astype(int)

    train['card1_TransactionAmt_mean'] = train.groupby(train['card1'].astype(np.float32))['TransactionAmt'].transform('mean')
    train['card1_TransactionAmt_std'] = train.groupby(train['card1'].astype(np.float32))['TransactionAmt'].transform('std')
    train['amt_vs_card_mean'] = train['TransactionAmt'] / (train['card1_TransactionAmt_mean'] + 1)

    if 'addr1' in train.columns and 'addr2' in train.columns:
        train['addr_match'] = (train['addr1'] == train['addr2']).astype(int)

    from sklearn.preprocessing import LabelEncoder
    cat_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain',
                'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
                'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29',
                'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38',
                'DeviceType', 'DeviceInfo',
                'P_emaildomain_bin', 'P_emaildomain_suffix',
                'R_emaildomain_bin', 'R_emaildomain_suffix']

    for col in cat_cols:
        if col in train.columns:
            le = LabelEncoder()
            train[col] = le.fit_transform(train[col].astype(str))

    train = train.copy()

    mlflow.log_param("email_features", 4)
    mlflow.log_param("time_features", 2)
    mlflow.log_param("freq_encoding_cols", 4)
    mlflow.log_param("amount_features", 3)
    mlflow.log_param("total_engineered_features", 13)
    mlflow.log_param("categorical_encoded_cols", len(cat_cols))
    print(f"Shape after Feature Engineering: {train.shape}")

/tmp/ipykernel_57/2362216575.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[c + '_bin'] = train[c].map(emails)
/tmp/ipykernel_57/2362216575.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[c + '_suffix'] = train[c].map(lambda x: str(x).split('.')[-1])
/tmp/ipykernel_57/2362216575.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a 

Shape after Feature Engineering: (590540, 450)
🏃 View run RandomForest_Feature_Engineering at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3/runs/959390ec7577407580601363cfb248df
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3


# 🎯 Feature Selection

In [12]:
with mlflow.start_run(run_name="RandomForest_Feature_Selection"):

    threshold = 0.90
    null_cols = [c for c in train.columns if train[c].isnull().sum() / len(train) > threshold]
    one_value_cols = [c for c in train.columns if train[c].nunique() <= 1]
    cols_to_drop = list(set(null_cols + one_value_cols))
    for c in ['isFraud', 'TransactionID', 'TransactionDT']:
        if c in cols_to_drop:
            cols_to_drop.remove(c)

    train.drop(cols_to_drop, axis=1, inplace=True)
    mlflow.log_param("cols_dropped_null_or_constant", len(cols_to_drop))

    from sklearn.ensemble import RandomForestClassifier

    X_fs = train.drop(['isFraud', 'TransactionID', 'TransactionDT'], axis=1, errors='ignore')
    y_fs = train['isFraud']

    sample_idx = X_fs.sample(frac=0.1, random_state=42).index
    X_sample = X_fs.loc[sample_idx]
    y_sample = y_fs.loc[sample_idx]

    quick_model = RandomForestClassifier(n_estimators=50, max_depth=8,
                                          n_jobs=-1, random_state=42)
    quick_model.fit(X_sample, y_sample)

    importances = pd.Series(quick_model.feature_importances_, index=X_fs.columns)
    selected_features = importances[importances > 0.001].index.tolist()
    removed_low_importance = len(X_fs.columns) - len(selected_features)

    mlflow.log_param("selection_method", "random_forest_importance_on_10pct_sample")
    mlflow.log_param("importance_threshold", 0.001)
    mlflow.log_param("features_removed_low_importance", removed_low_importance)
    mlflow.log_param("final_selected_features", len(selected_features))
    print(f"Selected {len(selected_features)} features out of {len(X_fs.columns)}")

Selected 192 features out of 447
🏃 View run RandomForest_Feature_Selection at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3/runs/70fc9915288144858a3292f3e8a3d6a8
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3


# 🚀 Training

In [13]:
X = train[selected_features]
y = train['isFraud']

split_idx = int(len(X) * 0.8)
X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]
print(f"Train: {len(X_train)} | Val: {len(X_val)}")

Train: 472432 | Val: 118108


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Underfit
with mlflow.start_run(run_name="RandomForest_Underfit"):
    params = {'n_estimators': 10, 'max_depth': 3, 'n_jobs': -1, 'random_state': 42}
    mlflow.log_params(params)
    mlflow.log_param("experiment_purpose", "underfitting_demonstration")

    m = RandomForestClassifier(**params)
    m.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, m.predict_proba(X_train)[:, 1])
    val_auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)
    print(f"[UNDERFIT] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc - val_auc:.4f}")

[UNDERFIT] Train: 0.8029 | Val: 0.8003 | Gap: 0.0026
🏃 View run RandomForest_Underfit at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3/runs/eab7ad94c1f04994a419af40c2d811c8
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3


In [16]:
# Overfit
with mlflow.start_run(run_name="RandomForest_Overfit"):
    params = {'n_estimators': 100, 'max_depth': 30, 'min_samples_leaf': 1,
              'n_jobs': -1, 'random_state': 42}
    mlflow.log_params(params)
    mlflow.log_param("experiment_purpose", "overfitting_demonstration")

    m = RandomForestClassifier(**params)
    m.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, m.predict_proba(X_train)[:, 1])
    val_auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)
    print(f"[OVERFIT] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc - val_auc:.4f}")


[OVERFIT] Train: 0.9902 | Val: 0.8908 | Gap: 0.0994
🏃 View run RandomForest_Overfit at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3/runs/1341546b93a84692a0af623e6a08342b
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3


In [17]:
# Best balanced
with mlflow.start_run(run_name="RandomForest_Best"):
    params = {'n_estimators': 300, 'max_depth': 15, 'min_samples_leaf': 10,
              'max_features': 'sqrt', 'class_weight': 'balanced',
              'n_jobs': -1, 'random_state': 42}
    mlflow.log_params(params)
    mlflow.log_param("experiment_purpose", "best_balanced_model")

    m = RandomForestClassifier(**params)
    m.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, m.predict_proba(X_train)[:, 1])
    val_auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)
    print(f"[BEST] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc - val_auc:.4f}")

[BEST] Train: 0.9518 | Val: 0.8871 | Gap: 0.0647
🏃 View run RandomForest_Best at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3/runs/d3cb7327d13d410d9584aa2237a64daa
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3


In [18]:
# Reduced overfit
with mlflow.start_run(run_name="RandomForest_ReducedOverfit"):
    params = {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 20,
              'max_features': 'sqrt', 'class_weight': 'balanced',
              'n_jobs': -1, 'random_state': 42}
    mlflow.log_params(params)
    mlflow.log_param("experiment_purpose", "reduce_overfit_gap")

    m = RandomForestClassifier(**params)
    m.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, m.predict_proba(X_train)[:, 1])
    val_auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)
    print(f"[REDUCED OVERFIT] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc - val_auc:.4f}")

[REDUCED OVERFIT] Train: 0.9022 | Val: 0.8757 | Gap: 0.0265
🏃 View run RandomForest_ReducedOverfit at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3/runs/3e9e7e635d954a4cba9d0725343f2a39
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3


# 💾 Pipeline & Model Registry

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
import mlflow.sklearn

best_params = {
    'n_estimators': 300, 'max_depth': 15, 'min_samples_leaf': 10,
    'max_features': 'sqrt', 'class_weight': 'balanced',
    'n_jobs': -1, 'random_state': 42
}

best_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=-999)),
    ('classifier', RandomForestClassifier(**best_params))
])

best_pipeline.fit(X_train, y_train)

with mlflow.start_run(run_name="RandomForest_Final_Pipeline"):
    train_auc = roc_auc_score(y_train, best_pipeline.predict_proba(X_train)[:, 1])
    val_auc = roc_auc_score(y_val, best_pipeline.predict_proba(X_val)[:, 1])

    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)
    mlflow.log_param("selected_features_count", len(selected_features))

    mlflow.sklearn.log_model(
        sk_model=best_pipeline,
        artifact_path="randomforest_pipeline",
        registered_model_name="RandomForest_Fraud_Pipeline",
        input_example=X_val.iloc[:5]
    )

    print(f"Pipeline saved to Model Registry!")
    print(f"Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f}")

2026/05/07 13:43:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 13:43:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can 

Pipeline saved to Model Registry!
Train AUC: 0.9520 | Val AUC: 0.8877
🏃 View run RandomForest_Final_Pipeline at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3/runs/7805d3842c114c43bf407c68cb182bf0
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/3
